# tbpp on Google Colab (free T4 GPU) — SIH 2026 PS 26158> **⚠️ UNEXECUTED — drafted, never run on a GPU; verify cell-by-cell on first use.**> The CPU pipeline is verified (see `docs/MULTI_VIDEO_RESULTS.md`). This notebook drafts the> GPU route toward the official target (<15 min for a 10-min video). Nothing in it has been> timed on a T4. If a cell fails, stop and fix that cell — do not skip ahead.**What it does:** clone tbpp → install → download a real drone video + GPS from the publicsources list → run keyframes → SfM → dense → open the viewer output.

In [ ]:
# 0. Check the GPU — expect a Tesla T4 on the free tier!nvidia-smi --query-gpu=name,memory.total --format=csvimport torchprint("torch cuda available:", torch.cuda.is_available())

In [ ]:
# 1. Clone the repo (branch luffy/sih26158-push) and install the package!git clone --branch luffy/sih26158-push https://github.com/Prateek2007-cmd/tbpp.git%cd tbpp!pip install -q -e .

In [ ]:
# 2. Download a demo video + GPS (KABR DJI_0679: 1080p, 143 s, DJI .SRT per-frame GPS)#    Any row from VIDEO_SOURCES.md works the same way — swap the two URLs.!mkdir -p /content/data && cd /content/data && \  wget -q "https://huggingface.co/datasets/imageomics/KABR-raw-videos/resolve/main/18_01_2023_session_5/DJI_0679.MP4" && \  wget -q "https://huggingface.co/datasets/imageomics/KABR-raw-videos/resolve/main/18_01_2023_session_5/DJI_0679.SRT"!ls -la /content/data/ && ffprobe -v error -show_entries format=duration -of csv /content/data/DJI_0679.MP4

In [ ]:
# 3. Keyframes (60 s window, frames 0-1800) — measured 198.8 s on CPU!python scripts/run_video_pipeline.py \    --video /content/data/DJI_0679.MP4 \    --config configs/kabr_dji0679.yaml \    --start-frame 0 --end-frame 1800 --stage keyframes

In [ ]:
# 4. SfM + SRT sync — measured ~71 min on CPU (global BA dominates dense 1080p).#    On Colab this is still CPU-bound unless COLMAP is built with CUDA (cell 6).#    Expected on the CPU path: 120/120 registered, reproj 0.83 px, SRT sync 120/120.!python scripts/run_video_pipeline.py \    --video /content/data/DJI_0679.MP4 \    --config configs/kabr_dji0679.yaml \    --start-frame 0 --end-frame 1800 \    --telemetry /content/data/DJI_0679.SRT --stage sfm

### 5. Dense — the GPU step (unverified)The CPU dense stage runs with `--stage dense`, but the speed target needs GPU MVS.COLMAP's CUDA dense path needs a CUDA-enabled COLMAP build; the pip `pycolmap` wheelis CPU-only. Two honest options:- **Option A (simplest, CPU dense):** run `--stage dense` as-is and time it — it works,  it is just slow. This is the verified path.- **Option B (GPU, unexecuted):** build COLMAP from source with CUDA on the Colab VM  (see the COLMAP docs; ~20–30 min build), then point the pipeline at that build.  **Nobody has done this yet for tbpp — the cell below is a starting point, not a  tested recipe.**

In [ ]:
# 5a. CPU dense (verified path, slow) — only if the GPS residual is under the 10 m gate.#     KABR DJI_0679 measured 16.2 m, so the gate REFUSES dense: that is correct behaviour,#     not a bug. For a dense demo on Colab, use the UAVLight slot or the city flyover instead.# !python scripts/run_video_pipeline.py --video /content/data/DJI_0679.MP4 \#     --config configs/kabr_dji0679.yaml --stage dense --run-dir data/outputs/<run_dir>

In [ ]:
# 5b. Dense demo that DOES pass the gate: the bundled city flyover (rendered, 20 s)!python scripts/run_video_pipeline.py \    --video data/city_flyover.mp4 --config configs/video_flyover.yaml

### 6. View the resultThe pipeline writes `viewer.html` into the run dir. Download the run dir (or the`city_textured.glb`) from the Colab file browser and open `viewer.html` in a browser,or serve it from Colab with the snippet below and use the public URL.

In [ ]:
# 6. Serve the viewer from Colab (one run dir at a time)import globrun = sorted(glob.glob("data/outputs/run_*"))[-1]print("latest run:", run)# !cd {run} && python -m http.server 8000 &   # then expose via ngrok / Colab port forwarding# Simpler: download the run dir from the file browser and open viewer.html locally.

### 7. Score it (UAVLight accuracy, optional)The UAVLight scorer needs the UAVLight scene download (~GBs) — do it only if you needthe accuracy numbers on Colab. The verified numbers are already in`data/outputs/run_20261116_231640/uavlight_score.json`: dense P@1m 84.8%, P@0.5m 72.8%,median 0.264 m, ATE median 0.089 m.